In [2]:
# Importa as ferramentas necessárias
from dotenv import load_dotenv  # Carrega variáveis do arquivo .env
import os  # Manipula variáveis de ambiente

from langchain_google_genai import ChatGoogleGenerativeAI  # Modelo Gemini via LangChain
from langchain_core.prompts import PromptTemplate  # Template de prompt
from langchain_core.output_parsers import StrOutputParser  # Parser de saída em texto
from langchain_community.tools.tavily_search import TavilySearchResults  # Ferramenta de busca web
from langchain_core.tools import tool  # Decorador para criar tools

In [4]:
# Carrega variáveis do .env e valida chaves
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError(
        "Variável GOOGLE_API_KEY não encontrada no arquivo .env. "
        "Adicione GOOGLE_API_KEY=SUA_CHAVE_AQUI"
    )
os.environ["GOOGLE_API_KEY"] = api_key

tavily_api_key = os.getenv("TAVILY_API_KEY")
if not tavily_api_key:
    raise RuntimeError(
        "Variável TAVILY_API_KEY não encontrada no arquivo .env. "
        "Adicione TAVILY_API_KEY=SUA_CHAVE_AQUI"
    )
os.environ["TAVILY_API_KEY"] = tavily_api_key

In [7]:
# Cria o modelo Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [9]:
# Define o template de prompt e monta a cadeia
modelo_de_prompt = PromptTemplate(
    template="Me diga quais os impactos da IA no assunto {assunto}?",
    input_variables=["assunto"],
)

cadeia = modelo_de_prompt | llm | StrOutputParser()

In [11]:
# Executa a cadeia com o assunto desejado
resposta = cadeia.invoke({"assunto": "Agricultura"})
print(resposta)

A Inteligência Artificial (IA) está revolucionando a agricultura, transformando-a de uma prática tradicional para um sistema mais inteligente, eficiente e sustentável. Os impactos são vastos e abrangem diversas áreas:

**1. Agricultura de Precisão e Otimização de Recursos:**

*   **Monitoramento Inteligente:** Drones equipados com câmeras multiespectrais, sensores no solo e imagens de satélite, combinados com IA, podem analisar a saúde das plantas, níveis de nutrientes, umidade do solo, infestações de pragas e doenças em tempo real e em nível granular (planta por planta).
*   **Otimização de Insumos:** A IA permite aplicar fertilizantes, pesticidas e água apenas onde e quando necessário, reduzindo drasticamente o desperdício, os custos e o impacto ambiental. Isso é conhecido como "pulverização localizada" ou "irrigação inteligente".
*   **Previsão de Colheitas:** Algoritmos de IA podem prever com maior precisão o rendimento das colheitas, ajudando os agricultores a planejar a logística

In [13]:
# Define uma ferramenta de busca web
@tool
def busca_web(query: str) -> list:
    """
    Busca na web por um termo específico
    """
    tavily_search = TavilySearchResults(max_results=5, search_depth="basic", max_tokens=1000)
    resultado_busca = tavily_search.invoke(query)
    return resultado_busca

In [15]:
# Exemplo de uso da ferramenta de busca
busca_web.invoke("IA na Agricultura")

[{'title': 'Como usar a Inteligência artificial na agricultura?',
  'url': 'https://www.senior.com.br/blog/inteligencia-artificial-na-agricultura-entenda-o-que-e',
  'content': 'A inteligência artificial na agricultura é a aplicação de algoritmos capazes de analisar grandes volumes de dados, aprender com eles e tomar decisões automaticamente. Na prática, isso significa contar com uma tecnologia que simula a lógica humana para prever, recomendar e agir, tudo com mais precisão e velocidade. [...] Uma das aplicações mais consolidadas da IA no campo está no monitoramento remoto da lavoura. Combinando imagens de satélite, drones agrícolas e sensores de solo, os sistemas com inteligência artificial conseguem identificar padrões que indicam pragas, doenças ou estresse hídrico em tempo real.\n\nEsses modelos utilizam aprendizado de máquina para detectar anomalias na vegetação e sugerir intervenções pontuais — o que evita perdas, reduz o uso de insumos e melhora a produtividade. [...] De acordo

In [20]:
tools = [busca_web]

In [21]:
llm_com_ferramenta = llm.bind(tools=tools)

In [22]:
# Define o template de prompt e monta a cadeia
modelo_de_prompt = PromptTemplate(
    template="Usando apenas as tools disponíveis me diga quais os impactos da IA no assunto {assunto}?",
    input_variables=["assunto"],
)

In [23]:
cadeia = modelo_de_prompt | llm_com_ferramenta | StrOutputParser()

In [24]:
# Executa a cadeia com o assunto desejado
resposta = cadeia.invoke({"assunto": "Arte"})
print(resposta)